Code to separeta tagged images in zooniverse in the different classes needed

In [ ]:
import pandas as pd
import sys
import os

In [ ]:
sys.path.insert(0, "../../")
from config import CROPPED_PATH
CSV_FILE_PATH = "/home/nicolas/Descargas/automating-allium-cepa-assay-analysis-with-ai-classifications.csv"
CROPS_PATH = os.path.join(CROPPED_PATH, 'classification')

In [18]:
df = pd.read_csv(CSV_FILE_PATH)

In [19]:
df.head()

,classification_id,user_name,user_id,user_ip,workflow_id,workflow_name,workflow_version,created_at,gold_standard,expert,metadata,annotations,subject_data,subject_ids
0,651783183,Nictauro,2892598.0,49301c690bcf4d3b3a13,29297,Mitosis Stage Classify,10.2,2025-07-09 12:47:14 UTC,NaN,NaN,"{""source"":""api"",""session"":""25b331a762b4ab010f1...","[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828150"":{""retired"":null,""Filename"":""A_15_...",110828150
1,651783233,Nictauro,2892598.0,49301c690bcf4d3b3a13,29297,Mitosis Stage Classify,10.2,2025-07-09 12:47:29 UTC,NaN,NaN,"{""source"":""api"",""session"":""25b331a762b4ab010f1...","[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828208"":{""retired"":null,""Filename"":""A_21_...",110828208
2,651783245,Nictauro,2892598.0,49301c690bcf4d3b3a13,29297,Mitosis Stage Classify,10.2,2025-07-09 12:47:33 UTC,NaN,NaN,"{""source"":""api"",""session"":""25b331a762b4ab010f1...","[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828153"":{""retired"":null,""Filename"":""A_15_...",110828153
3,651783364,Nictauro,2892598.0,49301c690bcf4d3b3a13,29297,Mitosis Stage Classify,10.2,2025-07-09 12:47:54 UTC,NaN,NaN,"{""source"":""api"",""session"":""25b331a762b4ab010f1...","[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828211"":{""retired"":null,""Filename"":""A_22_...",110828211
4,651783392,Nictauro,2892598.0,49301c690bcf4d3b3a13,29297,Mitosis Stage Classify,10.2,2025-07-09 12:47:59 UTC,NaN,NaN,"{""source"":""api"",""session"":""25b331a762b4ab010f1...","[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828203"":{""retired"":null,""Filename"":""A_20_...",110828203


In [20]:
annotations_subject_data = df[['annotations', 'subject_data']]
annotations_subject_data.head()

,annotations,subject_data
0,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828150"":{""retired"":null,""Filename"":""A_15_..."
1,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828208"":{""retired"":null,""Filename"":""A_21_..."
2,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828153"":{""retired"":null,""Filename"":""A_15_..."
3,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828211"":{""retired"":null,""Filename"":""A_22_..."
4,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828203"":{""retired"":null,""Filename"":""A_20_..."


In [21]:
import json

def extract_value(annotation_str):
    annotation_list = json.loads(annotation_str)
    return annotation_list[0]['value'] if annotation_list and 'value' in annotation_list[0] else None

# 2. Define a function to extract the filename
def extract_filename(subject_data_str):
    """
    Parses a JSON string from the 'subject_data' column
    and extracts the 'Filename' value from the nested dictionary.
    """
    if not isinstance(subject_data_str, str):
        return None
    try:
        # Load the string as a JSON object
        data = json.loads(subject_data_str)
        if not data:
            return None # Handles empty JSON object '{}'
            
        # The outer dictionary has a dynamic key. We get its value,
        # which is the inner dictionary.
        inner_dict = next(iter(data.values()))
        
        # Return the value of the 'Filename' key, or None if it doesn't exist
        return inner_dict.get('Filename')
    except (json.JSONDecodeError, StopIteration, AttributeError):
        # Handle cases where the string is not valid JSON,
        # or doesn't have the expected structure.
        return None

annotations_subject_data['stage'] = annotations_subject_data['annotations'].apply(extract_value)
annotations_subject_data['filename'] = annotations_subject_data['subject_data'].apply(extract_filename)
annotations_subject_data.head()

/tmp/ipykernel_231409/2245068582.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_subject_data['stage'] = annotations_subject_data['annotations'].apply(extract_value)
/tmp/ipykernel_231409/2245068582.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  annotations_subject_data['filename'] = annotations_subject_data['subject_data'].apply(extract_filename)


,annotations,subject_data,stage,filename
0,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828150"":{""retired"":null,""Filename"":""A_15_...",Prophase,A_15_10.png
1,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828208"":{""retired"":null,""Filename"":""A_21_...",Interphase,A_21_63.png
2,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828153"":{""retired"":null,""Filename"":""A_15_...",Prophase,A_15_23.png
3,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828211"":{""retired"":null,""Filename"":""A_22_...",Metaphase,A_22_43.png
4,"[{""task"":""T0"",""task_label"":""I which stage of m...","{""110828203"":{""retired"":null,""Filename"":""A_20_...",Anaphase,A_20_66.png


In [22]:
results = annotations_subject_data[['stage', 'filename']]
results.head()

,stage,filename
0,Prophase,A_15_10.png
1,Interphase,A_21_63.png
2,Prophase,A_15_23.png
3,Metaphase,A_22_43.png
4,Anaphase,A_20_66.png


In [23]:
cleaned_results = results.groupby('filename')['stage'].agg(lambda x: x.mode()[0]).reset_index()


# 3. Display the final, cleaned DataFrame
print("Cleaned DataFrame:")
cleaned_results.head()



Cleaned DataFrame:


,filename,stage
0,001_00026_86.png,Not a cell
1,001_00027_179.png,Chromosomal aberrations
2,001_00027_52.png,Not a cell
3,001_00027_55.png,Interphase
4,001_00027_80.png,Indeterminate


In [37]:
import os
import shutil
stage_to_filenames_dict = cleaned_results.groupby('stage')['filename'].apply(list).to_dict()

print("DataFrames were separated into a dictionary.\n")
print(f"Available stages: {list(stage_to_filenames_dict.keys())}")
print("-" * 35)

for stage in list(stage_to_filenames_dict.keys()):
    os.makedirs(os.path.join(CROPS_PATH, stage), exist_ok=True)
    for image in stage_to_filenames_dict[stage]:
        src = os.path.join(CROPS_PATH, image)
        dst = os.path.join(CROPS_PATH, stage, image)
        if os.path.exists(src):
            os.rename(src, dst)
            # shutil.copy2(src, dst)
            print(f"Moved {image} to {stage} folder.")
        else:
            print(f"Source file {src} does not exist. Skipping.")


DataFrames were separated into a dictionary.

Available stages: ['Anaphase', 'Chromosomal aberrations', 'Indeterminate', 'Interphase', 'Metaphase', 'Not a cell', 'Prophase', 'Telophase']
-----------------------------------
Moved 002_00058_24.png to Anaphase folder.
Moved 003_00002_8.png to Anaphase folder.
Moved 003_00009_106.png to Anaphase folder.
Moved 004_00051_122.png to Anaphase folder.
Moved 004_00070_113.png to Anaphase folder.
Moved A_10_72.png to Anaphase folder.
Moved A_115_137.png to Anaphase folder.
Moved A_12_82.png to Anaphase folder.
Moved A_12_84.png to Anaphase folder.
Moved A_14_18.png to Anaphase folder.
Moved A_14_42.png to Anaphase folder.
Moved A_14_83.png to Anaphase folder.
Moved A_15_27.png to Anaphase folder.
Moved A_17_3.png to Anaphase folder.
Moved A_17_80.png to Anaphase folder.
Moved A_17_85.png to Anaphase folder.
Moved A_17_89.png to Anaphase folder.
Moved A_17_96.png to Anaphase folder.
Moved A_20_66.png to Anaphase folder.
Moved A_21_9.png to Anaphas